# Exercise 57 — Missing data

## Concept

Missing values in pandas show up as `NaN` (for numeric columns) or `None`/`NaT` (for object/datetime). The unified detection method:

```python
df.isna()              # DataFrame of booleans, same shape
df["col"].isna().sum() # count of nulls in a column
df.notna()             # opposite
```

### Filling

```python
df["amount"].fillna(0)                # constant
df["amount"].fillna(df["amount"].mean())
df.fillna({"amount": 0, "city": "unknown"})    # per-column defaults
df["price"].ffill()                   # forward-fill (carry last known value)
df["price"].bfill()                   # backward-fill
```

### Dropping

```python
df.dropna()                           # drop rows with ANY null
df.dropna(subset=["amount"])          # drop rows where THIS column is null
df.dropna(axis="columns")             # drop COLUMNS with any null — rarely what you want
```

### `pd.NA` vs `np.nan`

Older pandas used `np.nan` (a float) universally — that's why an `int` column with one missing value becomes `float64`. Modern pandas has `pd.NA` for nullable integer/string/boolean dtypes (`"Int64"`, `"string"`, `"boolean"` — capital first letter). Use them when you need integer columns that can be null.

```python
s = pd.Series([1, 2, pd.NA], dtype="Int64")     # stays integer
```

In DE: **decide explicitly per column** whether nulls are an error, a default, or a meaningful state. Don't blanket-fillna or blanket-dropna — that erases information.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer": ["Alice", None, "Carol", "Bob", "Alice"],
    "amount":   [120.0, 45.0, np.nan, 89.0, np.nan],
    "region":   ["N", "S", "N", None, "E"],
})
print(df)


## Task 1 — Null counts and percentages

Return a DataFrame with columns `["column", "null_count", "null_pct"]`, one row per column, ordered by `null_pct` descending. The `null_pct` is a float between 0 and 100 (so 40.0 means 40%), rounded to 1 decimal.

```python
def null_report(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected for the setup df:
```
      column   null_count   null_pct
0     amount            2       40.0
1   customer            1       20.0
2     region            1       20.0
3   order_id            0        0.0
```

(Customer/region tie order is fine either way.)

In [ ]:
import pandas as pd

def null_report(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

report = null_report(df)
assert list(report.columns) == ["column", "null_count", "null_pct"]
assert report.iloc[0]["column"] == "amount"
assert report.iloc[0]["null_count"] == 2
assert report.iloc[0]["null_pct"] == 40.0
assert report.iloc[-1]["column"] == "order_id"
assert report.iloc[-1]["null_pct"] == 0.0
print("ok")


## Task 2 — Drop rows where a specific column is null

Return rows where `amount` is **not** null. Other null columns are fine — only `amount` must be present.

```python
def has_amount(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected: 3 rows (order_ids 1, 2, 4).

In [ ]:
import pandas as pd

def has_amount(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = has_amount(df)
assert result["order_id"].tolist() == [1, 2, 4]
print("ok")


## Task 3 — Fill with per-column defaults

Return a copy of `df` where:
- Null `customer` → `"unknown"`
- Null `amount` → `0.0`
- Null `region` → `"unspecified"`

Use a **single** `.fillna(...)` call with a dict.

```python
def fill_defaults(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

Expected: no nulls anywhere in the result.

In [ ]:
import pandas as pd

def fill_defaults(df: pd.DataFrame) -> pd.DataFrame:
    pass  # TODO

result = fill_defaults(df)
assert result.isna().sum().sum() == 0
assert result.loc[1, "customer"] == "unknown"
assert result.loc[2, "amount"] == 0.0
assert result.loc[3, "region"] == "unspecified"
print("ok")


## Task 4 — Mean imputation, but only when nulls are rare

Write `safe_mean_impute(df, column, max_null_pct)` that:
- If the column's null percentage is **<= `max_null_pct`** (a float 0–100), fill nulls with the column's mean and return the resulting DataFrame.
- Otherwise, raise `ValueError(f"too many nulls in {column}")` — too risky to impute without thinking.

```python
def safe_mean_impute(df: pd.DataFrame, column: str, max_null_pct: float) -> pd.DataFrame:
    ...
```

This pattern shows up constantly in DE: "impute is fine when sparse, but signal a problem when widespread".

Expected:
- `safe_mean_impute(df, "amount", 50)` succeeds (40% nulls ≤ 50%), fills with mean ≈ 84.67
- `safe_mean_impute(df, "amount", 30)` raises `ValueError`

In [ ]:
import pandas as pd

def safe_mean_impute(df: pd.DataFrame, column: str, max_null_pct: float) -> pd.DataFrame:
    pass  # TODO

result = safe_mean_impute(df, "amount", 50)
assert result["amount"].isna().sum() == 0
mean_amount = df["amount"].mean()
imputed = result.loc[df["amount"].isna(), "amount"]
assert (imputed == mean_amount).all()

try:
    safe_mean_impute(df, "amount", 30)
except ValueError as e:
    assert "amount" in str(e)
else:
    raise AssertionError("expected ValueError")
print("ok")
